<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/05_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL-00]
!pip install -q transformers[torch] datasets evaluate scikit-learn accelerate sentencepiece seaborn tqdm
print("✅ All required libraries installed successfully!")

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

print("📥 1. กำลังโหลดและทำ Undersampling ข้อมูล...")
df = pd.read_json("seed_dataset_final_train.jsonl", lines=True)

# คลีนข้อมูลเบื้องต้น
df['title'] = df['title'].fillna("")
df['content'] = df['content'].fillna("")
df['text'] = df['title'].astype(str) + " " + df['content'].astype(str)
df['text'] = df['text'].str.strip()

sentiment_map = {-1: 0, 0: 1, 1: 2}
df['label'] = df['sentiment'].map(sentiment_map)
df = df.dropna(subset=['label'])
df = df[df['text'] != ""]
df['label'] = df['label'].astype(int)
df = df[['text', 'label']].reset_index(drop=True)

# ---------------------------------------------------------
# 🌟 [จุดสำคัญ] กระบวนการ Undersampling (หั่นข้อมูลให้สมดุล)
# ---------------------------------------------------------
# แยก DataFrame ตามคลาส
df_neg = df[df['label'] == 0]
df_neu = df[df['label'] == 1]
df_pos = df[df['label'] == 2]

# ดูว่าคลาสที่น้อยที่สุด (Negative) มีกี่ข่าว
min_samples = len(df_neg)
print(f"📉 จำนวนข่าว Negative ทั้งหมด: {min_samples} ข่าว")

# สุ่มหั่นคลาส Neutral และ Positive ให้เหลือเท่ากับ Negative
df_neu_downsampled = resample(df_neu, replace=False, n_samples=min_samples, random_state=42)
df_pos_downsampled = resample(df_pos, replace=False, n_samples=min_samples, random_state=42)

# นำกลับมารวมกันเป็น DataFrame ตัวใหม่ที่สมดุลแล้ว (สัดส่วน 1:1:1)
df_balanced = pd.concat([df_neg, df_neu_downsampled, df_pos_downsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True) # สับไพ่ (Shuffle)

print("⚖️ สัดส่วนข้อมูลใหม่พร้อม Train:")
print(df_balanced['label'].value_counts())
# ---------------------------------------------------------

# แปลงเป็น Dataset และแบ่ง Train/Test (ตอนนี้ Test Set จะสมดุลไปด้วย)
hf_dataset = Dataset.from_pandas(df_balanced)
hf_dataset = hf_dataset.class_encode_column("label")
dataset = hf_dataset.train_test_split(test_size=0.15, seed=42, stratify_by_column='label')

print("\n🎯 สัดส่วนข้อมูลที่แบ่งได้:")
print(f"   - ชุดฝึกสอน (Train): {len(dataset['train'])} ข่าว")
print(f"   - ชุดทดสอบ (Test):  {len(dataset['test'])} ข่าว\n")

# คำนวณ Class Weights (ตอนนี้มันจะออกมาใกล้เคียง 1.0 ทุกคลาส เพราะเราทำสมดุลแล้ว)
train_labels = dataset['train']['label']
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("🔥 เตรียม Data เสร็จสิ้น พร้อมส่งไป Tokenize และ Train ต่อได้เลย!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

print("⚙️ กำลังโหลด Tokenizer และ Model จากโฟลเดอร์...")
# กำหนด Path ของโมเดลที่คุณทำ Domain Adaptation ไว้
model_path = r"C:\Users\USER\Desktop\FinDaptWangchanberta"

# โหลด Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# โหลด Model พร้อมระบุจำนวน Class (0=Negative, 1=Neutral, 2=Positive)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=3
)

# ฟังก์ชันสำหรับ Tokenize ข่าว
def tokenize_function(examples):
    # ปรับ max_length=400 (WangchanBERTa รองรับสูงสุด 416 tokens)
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=400
    )

print("📝 กำลัง Tokenize ข้อมูล (อาจใช้เวลาสักครู่)...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# ระบุให้ PyTorch รู้ว่าต้องใช้คอลัมน์ไหนบ้าง
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

print("✅ Tokenization เสร็จสมบูรณ์!")

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

# 1. ฟังก์ชันวัดผล (Evaluation Metrics)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # คำนวณ Accuracy และ Macro F1
    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro')

    return {"accuracy": acc, "macro_f1": macro_f1}

# 2. สร้าง Custom Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # ใช้ Weighted Cross-Entropy Loss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)

        # คำนวณ Loss
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 3. 🌟 ตั้งค่า Hyperparameters (แก้ไข eval_strategy แล้ว)
training_args = TrainingArguments(
    output_dir="./finbert_thai_results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",               # <--- แก้ไขตรงนี้ครับ
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    seed=42
)

# 4. เรียกใช้ WeightedTrainer
print("🚀 เตรียมพร้อมสำหรับการ Train...")
trainer = WeightedTrainer(
    class_weights=weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
)

# 5. เริ่ม Train!
print("🔥 เริ่มต้นกระบวนการ Fine-tuning...")
trainer.train()

# 6. บันทึกโมเดลที่เสร็จสมบูรณ์
final_model_path = r"C:\Users\USER\Desktop\Thai_FinBERT_Sentiment_Model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"🎉 Train เสร็จสิ้น! โมเดลถูกบันทึกไว้ที่: {final_model_path}")

In [ ]:
# สมมติว่ายังไม่ได้ปิดจอ และตัวแปร tokenized_datasets กับ trainer ยังอยู่
print("🔍 กำลังทดสอบโมเดลด้วยข้อมูล Test Set สมดุล...")
predictions = trainer.predict(tokenized_datasets["test"])

y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

target_names = ['Negative (0)', 'Neutral (1)', 'Positive (2)']
print("\n📊 ---------------- Classification Report ---------------- 📊")
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted Label (โมเดลทาย)')
plt.ylabel('True Label (ของจริง)')
plt.title('Confusion Matrix on Test Set')
plt.show()